# MNIST Classifiers – Comparison of Approaches (2025–2026 style)

We implement three different classifiers for the MNIST dataset using a common interface:

1. **Convolutional Neural Network (CNN)**
2. **Feed-Forward Neural Network (Simple MLP)**
3. **Random Forest**

All models follow the same abstract interface `MnistClassifierInterface`.

---

## 1. Imports & Utilities

In [1]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.ensemble import RandomForestClassifier
from abc import ABC, abstractmethod
import time

## 2. Model Definitions

In [2]:
class SimpleFFNN(nn.Module):
    """Simple 784-512-256-128-10 feed-forward network"""
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(784, 512),
            nn.ReLU(),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, 10)
        )

    def forward(self, x):
        return self.net(x)


class CNN(nn.Module):
    """Classical LeNet-like CNN for MNIST"""
    def __init__(self):
        super(CNN, self).__init__()
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, stride=1, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1)
        self.pool  = nn.MaxPool2d(2, 2)
        self.fc1   = nn.Linear(64*7*7, 128)
        self.fc2   = nn.Linear(128, 10)
        self.relu  = nn.ReLU()

    def forward(self, x):
        x = self.pool(self.relu(self.conv1(x)))
        x = self.pool(self.relu(self.conv2(x)))
        x = x.view(x.size(0), -1)
        x = self.relu(self.fc1(x))
        x = self.fc2(x)
        return x

## 3. Common Interface

In [3]:
class MnistClassifierInterface(ABC):
    @abstractmethod
    def train(self, X, y):
        pass

    @abstractmethod
    def predict(self, X):
        """Should always return numpy array of predicted class indices shape (N,)"""
        pass

## 4. Concrete Implementations

In [4]:
class CNNClassifier(MnistClassifierInterface):
    def __init__(self):
        self.model = CNN()
        print("CNN model created")

    def train(self, X_train, y_train, X_val=None, y_val=None, epochs=6, batch_size=128):
        print("Training CNN...")

        # reshape → (N, 1, 28, 28)
        X_train_t = torch.from_numpy(X_train).reshape(-1, 1, 28, 28).float()
        y_train_t = torch.from_numpy(y_train).long()

        if X_val is not None and y_val is not None:
            X_val_t   = torch.from_numpy(X_val).reshape(-1, 1, 28, 28).float()
            y_val_t   = torch.from_numpy(y_val).long()
            val_ds    = TensorDataset(X_val_t, y_val_t)
            val_loader = DataLoader(val_ds, batch_size=512, shuffle=False)
        else:
            val_loader = None

        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model.to(device)

        train_ds    = TensorDataset(X_train_t, y_train_t)
        train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)

        criterion = nn.CrossEntropyLoss()
        optimizer = optim.Adam(self.model.parameters(), lr=0.001)

        for epoch in range(epochs):
            self.model.train()
            running_loss = 0.0
            for images, labels in train_loader:
                images, labels = images.to(device), labels.to(device)
                optimizer.zero_grad()
                outputs = self.model(images)
                loss = criterion(outputs, labels)
                loss.backward()
                optimizer.step()
                running_loss += loss.item()

            print(f"Epoch {epoch+1:2d}  loss: {running_loss/len(train_loader):.5f}", end="")

            if val_loader is not None:
                correct, total = 0, 0
                self.model.eval()
                with torch.no_grad():
                    for images, labels in val_loader:
                        images, labels = images.to(device), labels.to(device)
                        outputs = self.model(images)
                        _, predicted = torch.max(outputs, 1)
                        total += labels.size(0)
                        correct += (predicted == labels).sum().item()
                print(f"  val acc: {100*correct/total:.2f}%")
            else:
                print()

    def predict(self, X):
        """
        Predicts classes for input X.
        Accepts:
          - (N, 784)
          - (N, 28, 28)
          - (N, 1, 28, 28)
        Returns: np.ndarray shape (N,) with class indices
        """
        if isinstance(X, np.ndarray):
            X_t = torch.from_numpy(X).float()
        else:
            X_t = torch.as_tensor(X, dtype=torch.float32)

        # Normalize shape → (N, 1, 28, 28)
        if X_t.dim() == 2:
            X_t = X_t.view(-1, 1, 28, 28)
        elif X_t.dim() == 3:
            X_t = X_t.unsqueeze(1)
        elif X_t.dim() != 4 or X_t.shape[1] not in (1,):
            raise ValueError(f"Unsupported input shape: {X_t.shape}")

        device = next(self.model.parameters()).device
        self.model.eval()

        predictions = []
        with torch.inference_mode():
            X_t = X_t.to(device)
            loader = DataLoader(TensorDataset(X_t), batch_size=256, shuffle=False)

            for batch in loader:
                logits = self.model(batch[0])
                preds = torch.argmax(logits, dim=1)
                predictions.append(preds.cpu())

        return torch.cat(predictions).numpy()

In [5]:
class NeuralNetClassifier(MnistClassifierInterface):
    def __init__(self):
        self.model = SimpleFFNN()
        print("MLP (feed-forward NN) model created")

    def train(self, X_train, y_train, epochs=12, batch_size=128):
        print("Training MLP...")
        X_t = torch.from_numpy(X_train).float()
        y_t = torch.from_numpy(y_train).long()

        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model.to(device)

        loader = DataLoader(TensorDataset(X_t, y_t), batch_size=batch_size, shuffle=True)

        criterion = nn.CrossEntropyLoss()
        optimizer = optim.Adam(self.model.parameters(), lr=0.001)

        for epoch in range(epochs):
            running_loss = 0.0
            for xb, yb in loader:
                xb, yb = xb.to(device), yb.to(device)
                optimizer.zero_grad()
                out = self.model(xb)
                loss = criterion(out, yb)
                loss.backward()
                optimizer.step()
                running_loss += loss.item()
            print(f"Epoch {epoch+1:2d}  loss: {running_loss/len(loader):.5f}")

    def predict(self, X):
        if isinstance(X, np.ndarray):
            X_t = torch.from_numpy(X).float()
        else:
            X_t = torch.as_tensor(X, dtype=torch.float32)

        if X_t.dim() != 2:
            raise ValueError("MLP expects flat input (N, 784)")

        device = next(self.model.parameters()).device
        self.model.eval()

        with torch.inference_mode():
            X_t = X_t.to(device)
            if X_t.shape[0] > 20000:
                loader = DataLoader(TensorDataset(X_t), batch_size=512)
                preds = []
                for b in loader:
                    out = self.model(b[0])
                    preds.append(torch.argmax(out, dim=1).cpu())
                return torch.cat(preds).numpy()
            else:
                out = self.model(X_t)
                return torch.argmax(out, dim=1).cpu().numpy()

In [6]:
class MnistRandomForestClassifier(MnistClassifierInterface):
    def __init__(self, n_estimators=120):
        print("Random Forest model created")
        self.model = RandomForestClassifier(
            n_estimators=n_estimators,
            random_state=42,
            n_jobs=-1
        )

    def train(self, X_train, y_train):
        print("Training Random Forest...")
        start = time.time()
        self.model.fit(X_train, y_train)
        print(f"Training finished in {time.time()-start:.1f} seconds")

    def predict(self, X):
        print("Predicting with Random Forest...")
        return self.model.predict(X)

## 5. Wrapper

In [7]:
class MnistClassifier:
    def __init__(self, algorithm: str):
        algorithm = algorithm.lower()
        if algorithm == "cnn":
            self.model = CNNClassifier()
        elif algorithm in ("nn", "mlp"):
            self.model = NeuralNetClassifier()
        elif algorithm in ("rf", "randomforest"):
            self.model = MnistRandomForestClassifier()
        else:
            raise ValueError("Supported algorithms: 'cnn', 'nn'/'mlp', 'rf'/'randomforest'")

    def train(self, X, y, **kwargs):
        self.model.train(X, y, **kwargs)

    def predict(self, X):
        return self.model.predict(X)

## 6. Data Loading & Splitting

In [8]:
# Load MNIST
mnist = fetch_openml('mnist_784', version=1, as_frame=False, cache=True)
X = mnist.data.astype(np.float32) / 255.0
y = mnist.target.astype(np.int64)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train: {X_train.shape}, Test: {X_test.shape}")

Train: (56000, 784), Test: (14000, 784)


## 7. Training & Evaluation Examples

In [9]:
# ─── CNN ───────────────────────────────────────────────
cnn = MnistClassifier("cnn")
cnn.train(X_train, y_train, epochs=7)

pred_cnn = cnn.predict(X_test)
acc_cnn  = accuracy_score(y_test, pred_cnn)
print(f"CNN test accuracy: {acc_cnn:.4f} ({acc_cnn*100:.2f}%)")

CNN model created
Training CNN...
Epoch  1  loss: 0.24923
Epoch  2  loss: 0.06266
Epoch  3  loss: 0.04288
Epoch  4  loss: 0.03288
Epoch  5  loss: 0.02476
Epoch  6  loss: 0.02024
Epoch  7  loss: 0.01667
CNN test accuracy: 0.9883 (98.83%)


In [10]:
# ─── MLP ───────────────────────────────────────────────
mlp = MnistClassifier("nn")
mlp.train(X_train, y_train, epochs=14)

pred_mlp = mlp.predict(X_test)
acc_mlp  = accuracy_score(y_test, pred_mlp)
print(f"MLP test accuracy: {acc_mlp:.4f} ({acc_mlp*100:.2f}%)")

MLP (feed-forward NN) model created
Training MLP...
Epoch  1  loss: 0.31997
Epoch  2  loss: 0.11038
Epoch  3  loss: 0.07136
Epoch  4  loss: 0.05034
Epoch  5  loss: 0.03722
Epoch  6  loss: 0.03148
Epoch  7  loss: 0.02172
Epoch  8  loss: 0.02236
Epoch  9  loss: 0.01838
Epoch 10  loss: 0.01722
Epoch 11  loss: 0.01424
Epoch 12  loss: 0.01495
Epoch 13  loss: 0.01003
Epoch 14  loss: 0.01109
MLP test accuracy: 0.9801 (98.01%)


In [11]:
# ─── Random Forest ─────────────────────────────────────
rf = MnistClassifier("rf")
rf.train(X_train, y_train)

pred_rf = rf.predict(X_test)
acc_rf  = accuracy_score(y_test, pred_rf)
print(f"Random Forest test accuracy: {acc_rf:.4f} ({acc_rf*100:.2f}%)")

Random Forest model created
Training Random Forest...
Training finished in 7.3 seconds
Predicting with Random Forest...
Random Forest test accuracy: 0.9674 (96.74%)


## 8. Test

In [18]:

print(f"Prediction: {pred_cnn[2].reshape(1, -1)[0][0]}")
print(f"truth: {y_test[2]}")

Prediction: 1
truth: 1


In [21]:
print(f"Prediction: {pred_rf[5].reshape(1, -1)[0][0]}")
print(f"truth: {y_test[5]}")

Prediction: 5
truth: 5


In [23]:
print(f"Prediction: {pred_mlp[10].reshape(1, -1)[0][0]}")
print(f"truth: {y_test[10]}")

Prediction: 6
truth: 6
